# Beginner Tutorial 1: Your First Workflow

## What You Will Learn

By the end of this notebook you will:

- Understand what Scalable is and why it exists
- Know what Dask is (the engine under the hood)
- Create a manifest file (`scalable.yaml`)
- Validate, plan, and run a workflow end-to-end
- Read telemetry output to understand what happened

## Prerequisites

- Python 3.11+
- Basic Python knowledge (functions, imports)
- NO distributed computing experience needed!

In [1]:
# Install Scalable (skip if already installed)
# !pip install scalable

In [1]:
# Verify installation
import scalable
print(f"Scalable version: {scalable.__version__}")

Scalable version: 2.0.0


## 💡 Key Concept: What is a Workflow?

A **workflow** is a sequence of computational steps that transforms inputs into outputs.
Think of it like a recipe:
- **Ingredients** = your input data
- **Steps** = your Python functions
- **Final dish** = your results

## 💡 Key Concept: What is Distributed Computing?

**Distributed computing** means splitting work across multiple processors or computers.
Instead of one CPU doing 1000 tasks one-by-one, you have 10 CPUs each doing 100 tasks simultaneously.

**Analogy:** Stuffing 1000 envelopes alone takes hours. With 10 friends helping,
each person stuffs 100 and you finish 10× faster.

## 💡 Key Concept: What is Dask?

**Dask** is a Python library for parallel computing — it's the "engine" Scalable uses.
Scalable adds workflow management, manifests, caching, and telemetry on top of Dask.

Why Dask? It:
- Integrates with NumPy/pandas
- Scales from laptop to 1000s of nodes
- Has a mature task scheduler
- Is widely used in scientific computing

## Step 1: Create a Project Directory

Scalable workflows live in a dedicated directory with a manifest file.

In [2]:
import os
import tempfile

# Create a temporary project directory for this tutorial
# Explanation: We use a temp directory so this notebook doesn't leave files on your system
project_dir = tempfile.mkdtemp(prefix="scalable-beginner-01-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

Working directory: /var/folders/8c/787g_pzs7wq7hljnnx3prdzr0000gn/T/scalable-beginner-01-27jmo9up


## 💡 Key Concept: Declarative Programming

The manifest is **declarative** — you describe WHAT you want, not HOW to achieve it.

**Imperative** (how): "SSH into server, run this command, check output, allocate memory..."

**Declarative** (what): "I need 2 workers with 1 CPU and 1GB RAM each."

Scalable figures out the "how" for you. This is the same philosophy behind:
- SQL (`SELECT name FROM users` — you say what data, not how to fetch it)
- HTML (`<h1>Title</h1>` — you say what it is, not how to render it)

## Step 2: Write a Manifest

The manifest (`scalable.yaml`) is the single source of truth for your workflow.
It answers:
- **What** is this project? (name)
- **Where** should it run? (targets)
- **How much** resources? (components)
- **What** work units? (tasks)

In [3]:
# Write the manifest file
# Explanation: Each section has a specific purpose (explained in comments)
manifest_content = """\
version: 1                    # Schema version (always 1 for now)
project:
  name: hello-scalable        # Human-readable project name

targets:                      # WHERE code runs
  local:                      # Target name (we'll use this later)
    provider: local           # Use the local machine
    max_workers: 2            # Run 2 workers in parallel
    threads_per_worker: 1     # 1 thread per worker
    processes: false           # Use threads (fast startup)
    containers: none          # No containerization

components:                   # HOW MUCH resources each piece needs
  analysis:                   # Component name
    cpus: 1                   # 1 CPU per worker
    memory: 1G               # 1 GB RAM per worker

tasks:                        # WHAT work units exist
  run_analysis:               # Task name
    component: analysis       # Links to the component above
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest_content)

print("Manifest written to scalable.yaml")
print("---")
print(manifest_content)

Manifest written to scalable.yaml
---
version: 1                    # Schema version (always 1 for now)
project:
  name: hello-scalable        # Human-readable project name

targets:                      # WHERE code runs
  local:                      # Target name (we'll use this later)
    provider: local           # Use the local machine
    max_workers: 2            # Run 2 workers in parallel
    threads_per_worker: 1     # 1 thread per worker
    processes: false           # Use threads (fast startup)
    containers: none          # No containerization

components:                   # HOW MUCH resources each piece needs
  analysis:                   # Component name
    cpus: 1                   # 1 CPU per worker
    memory: 1G               # 1 GB RAM per worker

tasks:                        # WHAT work units exist
  run_analysis:               # Task name
    component: analysis       # Links to the component above



## Step 3: Validate the Manifest

**Validation** = checking that your configuration is correct BEFORE running.
It catches typos, missing fields, and invalid values early.

In [4]:
from scalable.manifest.parser import load_manifest
from scalable.manifest.validate import validate_manifest

# Explanation: load_manifest reads the YAML file into a ManifestModel object
# Explanation: validate_manifest checks it against the schema rules
manifest = load_manifest("./scalable.yaml")
report = validate_manifest(manifest)

if not report.ok:
    for issue in report.errors:
        print(f"ERROR: {issue.path}: {issue.message}")
else:
    print("✓ Manifest is valid (0 errors, 0 warnings)")

✓ Manifest is valid (0 errors, 0 warnings)


## Step 4: Write Workflow Code

Now let's write the Python function that does actual work.

### 💡 Key Concept: Futures

A **future** is a promise of a result that hasn't been computed yet.
When you call `session.submit()`, the task starts in the background and you
get a future immediately. Later, `session.gather()` waits for all results.

**Analogy:** Ordering food at a counter — you get a receipt number (future)
immediately. The food is being prepared in the background. When your number
is called, you pick up your food (gather the result).

In [5]:
import time
from scalable import ScalableSession


def analyze_scenario(scenario_id: int) -> dict:
    """Simulate an analysis task.
    
    In a real workflow this might run a climate model, process satellite data,
    or train an ML model. Here we just simulate work with a sleep.
    """
    time.sleep(0.5)  # Simulate 0.5 seconds of computation
    return {
        "scenario_id": scenario_id,
        "result": scenario_id * 42,
        "status": "complete",
    }

In [6]:
# Create a session from our manifest
# Explanation: ScalableSession sets up the Dask cluster based on our manifest
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
plan = session.plan()
client = session.start(plan)

# Submit 6 tasks to be executed in parallel
# Explanation: submit() returns immediately with a future (the work runs in background)
futures = []
for i in range(6):
    future = client.submit(analyze_scenario, i, tag="analysis")
    futures.append(future)

print(f"Submitted {len(futures)} tasks")
print("Tasks are running in the background on 2 workers...")

Submitted 6 tasks
Tasks are running in the background on 2 workers...


In [7]:
# Gather results (blocks until all tasks complete)
# Explanation: gather() waits for all futures to finish and returns results
results = client.gather(futures)

print(f"\nCompleted {len(results)} scenarios!")
for r in results:
    print(f"  Scenario {r['scenario_id']}: result = {r['result']}")


Completed 6 scenarios!
  Scenario 0: result = 0
  Scenario 1: result = 42
  Scenario 2: result = 84
  Scenario 3: result = 126
  Scenario 4: result = 168
  Scenario 5: result = 210


## 🤔 Think About It

With 6 tasks and 2 workers, each taking 0.5 seconds:
- **Sequential** (no parallelism): 6 × 0.5s = 3.0 seconds
- **Parallel with 2 workers**: 3 batches × 0.5s = ~1.5 seconds

The speedup is approximately 2× with 2 workers. This is the fundamental
value of distributed computing — trading more hardware for less time.

In [8]:
# Clean up the session
session.close()
print("Session closed.")

Session closed.


## 📖 Vocabulary Summary

| Term | Definition |
|------|------------|
| Workflow | Sequence of computational steps (inputs → functions → outputs) |
| Distributed Computing | Splitting work across multiple processors/computers |
| Dask | Python parallel computing library (Scalable's engine) |
| Manifest | Declarative config file describing desired state |
| Declarative Programming | Describing WHAT you want, not HOW to do it |
| Provider | Abstraction over execution backend (local, HPC, cloud) |
| Worker | Process/thread that executes tasks |
| Future | Placeholder for a result being computed asynchronously |
| Validation | Checking correctness before execution |

## Next Steps

- **Next notebook:** `02_manifest_system.ipynb` — deep dive into YAML and declarative configuration
- **Standard notebook:** `../01_getting_started.ipynb` — same topic, less explanation
- **Try:** Change `max_workers` to 4 in the manifest and re-run. Is it faster?

In [9]:
# Cleanup: remove temporary directory
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")

Cleaned up /var/folders/8c/787g_pzs7wq7hljnnx3prdzr0000gn/T/scalable-beginner-01-27jmo9up
